In [83]:
import requests
import re
import json
import pandas as pd
import time

In [85]:
response = requests.get('https://www.smogon.com/dex/sv/pokemon/')
match = re.search(r'dexSettings = (\{.+\})</script>', response.text, re.DOTALL)
data = json.loads(match.group(1))
basics = data['injectRpcs'][1][1]

# Build move lookup
move_lookup = {m['name']: m for m in basics['moves']}
print(f"Move lookup: {len(move_lookup)} moves")

# Build Smogon Pokemon list
pokemon_list_smogon = basics['pokemon']
smogon_data = []

for p in pokemon_list_smogon:
    if p.get('isNonstandard') != 'Standard':
        continue
    if p.get('oob') is None:
        continue
        
    smogon_data.append({
        'name': p['name'],
        'dex_number': p['oob']['dex_number'],
        'tier': p['formats'][0] if p['formats'] else 'Untiered',
        'abilities': ', '.join(p['abilities']),
        'types': ', '.join(p['types']),
        'is_fully_evolved': len(p['oob']['evos']) == 0
    })

df_smogon = pd.DataFrame(smogon_data)
df_smogon['name'] = df_smogon['name'].str.lower().str.replace(' ', '-')
df_smogon = df_smogon.drop_duplicates(subset='name', keep='first')
print(f"Smogon Pokemon: {len(df_smogon)}")

Move lookup: 937 moves
Smogon Pokemon: 862


In [87]:
full_mapping = {
    'tornadus': 'tornadus-incarnate',
    'thundurus': 'thundurus-incarnate',
    'landorus': 'landorus-incarnate',
    'enamorus': 'enamorus-incarnate',
    'minior': 'minior-red-meteor',
    'mimikyu': 'mimikyu-disguised',
    'eiscue': 'eiscue-ice',
    'morpeko': 'morpeko-full-belly',
    'toxtricity': 'toxtricity-amped',
    'urshifu': 'urshifu-single-strike',
    'lycanroc': 'lycanroc-midday',
    'indeedee': 'indeedee-male',
    'indeedee-f': 'indeedee-female',
    'basculin': 'basculin-red-striped',
    'oricorio': 'oricorio-baile',
    "oricorio-pa'u": 'oricorio-pau',
    'meowstic-m': 'meowstic-male',
    'meowstic-f': 'meowstic-female',
    'pyroar': 'pyroar-male',
    'palafin': 'palafin-zero',
    'dudunsparce': 'dudunsparce-two-segment',
    'maushold': 'maushold-family-of-four',
    'maushold-four': 'maushold-family-of-three',
    'squawkabilly': 'squawkabilly-green-plumage',
    'squawkabilly-blue': 'squawkabilly-blue-plumage',
    'squawkabilly-yellow': 'squawkabilly-yellow-plumage',
    'squawkabilly-white': 'squawkabilly-white-plumage',
    'tatsugiri': 'tatsugiri-curly',
    'deoxys': 'deoxys-normal',
    'giratina': 'giratina-altered',
    'shaymin': 'shaymin-land',
    'keldeo': 'keldeo-ordinary',
    'meloetta': 'meloetta-aria',
    'basculegion': 'basculegion-male',
    'basculegion-f': 'basculegion-female',
    'oinkologne': 'oinkologne-male',
    'oinkologne-f': 'oinkologne-female',
    'greninja-bond': 'greninja-battle-bond',
    'zacian-crowned': 'zacian-crowned',
    'zamazenta-crowned': 'zamazenta-crowned',
    'calyrex-ice': 'calyrex-ice',
    'calyrex-shadow': 'calyrex-shadow',
    'necrozma-dawn-wings': 'necrozma-dawn',
    'necrozma-dusk-mane': 'necrozma-dusk',
    'rockruff-dusk': 'rockruff-own-tempo',
    'ogerpon-cornerstone': 'ogerpon-cornerstone-mask',
    'ogerpon-hearthflame': 'ogerpon-hearthflame-mask',
    'ogerpon-wellspring': 'ogerpon-wellspring-mask',
    'tauros-paldea-aqua': 'tauros-paldea-aqua-breed',
    'tauros-paldea-blaze': 'tauros-paldea-blaze-breed',
    'tauros-paldea-combat': 'tauros-paldea-combat-breed',
    'sinistea-antique': 'sinistea',
    'polteageist-antique': 'polteageist',
    'poltchageist-artisan': 'poltchageist',
    'sinistcha-masterpiece': 'sinistcha',
}

df_smogon['name'] = df_smogon['name'].replace(full_mapping)
print("Name standardization complete")

Name standardization complete


In [90]:
pokemon_list = []

for i in range(1, 1026):
    response = requests.get(f"https://pokeapi.co/api/v2/pokemon/{i}")
    data = response.json()
    species_id = data['species']['url'].split('/')[-2]
    
    species = requests.get(f"https://pokeapi.co/api/v2/pokemon-species/{species_id}/")
    species_data = species.json()
    
    stats = {s['stat']['name']: s['base_stat'] for s in data['stats']}
    types = [t['type']['name'] for t in data['types']]
    
    pokemon_list.append({
        'id': data['id'],
        'name': data['name'],
        'type1': types[0],
        'type2': types[1] if len(types) > 1 else None,
        'hp': stats['hp'],
        'attack': stats['attack'],
        'defense': stats['defense'],
        'sp_attack': stats['special-attack'],
        'sp_defense': stats['special-defense'],
        'speed': stats['speed'],
        'is_legendary': species_data['is_legendary'],
        'is_mythical': species_data['is_mythical']
    })
    
    time.sleep(0.2)

print(f"Scraped {len(pokemon_list)} base Pokemon")

Scraped 1025 base Pokemon


In [92]:
df_base = pd.DataFrame(pokemon_list)

unmatched = df_smogon[~df_smogon['name'].isin(df_base['name'])]
alternate_forms = []

for name in unmatched['name'].tolist():
    time.sleep(0.5)
    response = requests.get(f"https://pokeapi.co/api/v2/pokemon/{name}")
    if response.status_code == 200:
        data = response.json()
        stats = {s['stat']['name']: s['base_stat'] for s in data['stats']}
        types = [t['type']['name'] for t in data['types']]
        species_id = data['species']['url'].split('/')[-2]
        
        time.sleep(0.5)
        species = requests.get(f"https://pokeapi.co/api/v2/pokemon-species/{species_id}/")
        species_data = species.json()
        
        alternate_forms.append({
            'id': data['id'],
            'name': name,
            'type1': types[0],
            'type2': types[1] if len(types) > 1 else None,
            'hp': stats['hp'],
            'attack': stats['attack'],
            'defense': stats['defense'],
            'sp_attack': stats['special-attack'],
            'sp_defense': stats['special-defense'],
            'speed': stats['speed'],
            'is_legendary': species_data['is_legendary'],
            'is_mythical': species_data['is_mythical']
        })

print(f"Scraped {len(alternate_forms)} alternate forms")

Scraped 98 alternate forms


In [94]:
arceus_base = df_base[df_base['name'] == 'arceus'].iloc[0]

arceus_types = [
    'bug', 'dark', 'dragon', 'electric', 'fighting', 'fire',
    'flying', 'ghost', 'grass', 'ground', 'ice', 'poison',
    'psychic', 'rock', 'steel', 'water', 'fairy'
]

arceus_forms = []
for t in arceus_types:
    form = arceus_base.to_dict()
    form['name'] = f'arceus-{t}'
    form['type1'] = t
    form['type2'] = None
    arceus_forms.append(form)

print(f"Created {len(arceus_forms)} Arceus forms")

Created 17 Arceus forms


In [96]:
df_alternate = pd.DataFrame(alternate_forms)
df_arceus = pd.DataFrame(arceus_forms)

df_stats = pd.concat([df_base, df_alternate, df_arceus], ignore_index=True)
df_stats = df_stats.drop_duplicates(subset='name', keep='first')

df_merged = pd.merge(df_stats, df_smogon, on='name', how='inner')
df_merged = df_merged.drop_duplicates(subset='name', keep='first')

print(f"Merged dataset: {len(df_merged)} Pokemon")

Merged dataset: 848 Pokemon


In [98]:
offensive_priority_moves = [
    'Extreme Speed', 'Bullet Punch', 'Mach Punch', 'Ice Shard',
    'Aqua Jet', 'Shadow Sneak', 'Sucker Punch', 'Accelerock',
    'First Impression', 'Jet Punch', 'Water Shuriken', 'Quick Attack',
    'Vacuum Wave', 'Thunderclap', 'Upper Hand'
]

def get_sv_moves(pokemon_name, move_lookup):
    response = requests.get(f'https://pokeapi.co/api/v2/pokemon/{pokemon_name}')
    if response.status_code != 200:
        return None
    
    data = response.json()
    sv_moves = []
    for move_entry in data['moves']:
        for version in move_entry['version_group_details']:
            if version['version_group']['name'] == 'scarlet-violet':
                move_name = move_entry['move']['name'].replace('-', ' ').title()
                if move_name in move_lookup:
                    sv_moves.append(move_lookup[move_name])
                break
    
    return sv_moves

def extract_move_features(moves):
    if not moves:
        return None
    
    has_recovery = any('Heal' in m.get('flags', []) for m in moves)
    has_setup = any('Snatch' in m.get('flags', []) for m in moves)
    has_priority = any(m['name'] in offensive_priority_moves for m in moves)
    has_pivot = any(m['name'] in ['U-Turn', 'Volt Switch', 'Flip Turn', 'Parting Shot'] for m in moves)
    has_hazards = any(m['name'] in ['Stealth Rock', 'Spikes', 'Toxic Spikes', 'Sticky Web'] for m in moves)
    has_status = any(m['name'] in ['Thunder Wave', 'Toxic', 'Will-O-Wisp', 'Spore', 'Sleep Powder'] for m in moves)
    total_moves = len(moves)
    coverage_types = len(set(m['type'] for m in moves if m.get('category') != 'Non-Damaging'))
    avg_power = sum(m.get('power', 0) for m in moves if m.get('power', 0) > 0) / max(1, sum(1 for m in moves if m.get('power', 0) > 0))
    
    return {
        'has_recovery': has_recovery,
        'has_setup': has_setup,
        'has_priority': has_priority,
        'has_pivot': has_pivot,
        'has_hazards': has_hazards,
        'has_status': has_status,
        'total_moves': total_moves,
        'coverage_types': coverage_types,
        'avg_power': round(avg_power, 2)
    }

print("Functions defined")

Functions defined


In [100]:
move_features_list = []

for i, row in df_merged.iterrows():
    name = row['name']
    api_name = 'arceus' if name.startswith('arceus-') else name
    
    moves = get_sv_moves(api_name, move_lookup)
    
    if moves is not None:
        features = extract_move_features(moves)
        if features is not None:
            features['name'] = name
            move_features_list.append(features)
        else:
            print(f"✗ No features: {name}")
    else:
        print(f"✗ Failed: {name}")
    
    time.sleep(0.3)
    
    if i % 50 == 0:
        print(f"Progress: {i}/{len(df_merged)}")

print(f"\nDone! Scraped {len(move_features_list)} Pokemon")

Progress: 0/848
Progress: 50/848
Progress: 100/848
Progress: 150/848
Progress: 200/848
Progress: 250/848
Progress: 300/848
Progress: 350/848
Progress: 400/848
Progress: 450/848
Progress: 500/848
Progress: 550/848
Progress: 600/848
Progress: 650/848
Progress: 700/848
Progress: 750/848
✗ No features: greninja-battle-bond
Progress: 800/848
Progress: 850/848

Done! Scraped 847 Pokemon


In [102]:
df_moves = pd.DataFrame(move_features_list)

df_final = pd.merge(df_merged, df_moves, on='name', how='inner')
df_final = df_final.drop_duplicates(subset='name', keep='first')

df_stats.to_csv('pokemon_data.csv', index=False)
df_smogon.to_csv('smogon_data.csv', index=False)
df_merged.to_csv('merged_data.csv', index=False)
df_moves.to_csv('move_data.csv', index=False)
df_final.to_csv('pokemon_movepool_data.csv', index=False)

print(f"pokemon_data.csv: {len(df_stats)}")
print(f"smogon_data.csv: {len(df_smogon)}")
print(f"merged_data.csv: {len(df_merged)}")
print(f"move_data.csv: {len(df_moves)}")
print(f"pokemon_movepool_data.csv: {len(df_final)}")

pokemon_data.csv: 1140
smogon_data.csv: 862
merged_data.csv: 848
move_data.csv: 847
pokemon_movepool_data.csv: 847
